# EigenAlpha Phase 0 — Exploratory Data Analysis
*Nifty 500 Factor Research | EigenAlpha v0.1.0*

This notebook runs the full EDA pipeline: data ingestion, PCA decomposition, K-Means clustering, 
and factor IC analysis.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import *
from data.loader import DataLoader
from data.preprocessor import Preprocessor
from factors.engine import FactorEngine
from factors.ic_analysis import InformationCoefficient
from pca.decompose import CovarianceDecomposer
from pca.cluster import MarketClusterer

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print("EigenAlpha Phase 0 \u2014 EDA Notebook")
print(f"Universe: {len(NIFTY500_TICKERS)} tickers | Period: {START_DATE} to {END_DATE}")

## 1. Data Ingestion

In [ ]:
loader = DataLoader()

# Try cached data first
import os
cache_path = os.path.join('..', 'data_cache', 'nifty500_ohlcv.parquet')
try:
    raw = loader.load(cache_path)
    print(f'Loaded cached data: {raw.shape}')
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close'] if 'Close' in raw.columns.get_level_values(0) else raw
        volumes = raw['Volume'] if 'Volume' in raw.columns.get_level_values(0) else pd.DataFrame()
    else:
        prices = raw
        volumes = pd.DataFrame()
except FileNotFoundError:
    print('Cache not found. Downloading (this may take a few minutes)...')
    raw = loader.fetch(tickers=NIFTY500_TICKERS, start=START_DATE, end=END_DATE)
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close'] if 'Close' in raw.columns.get_level_values(0) else raw
        volumes = raw['Volume'] if 'Volume' in raw.columns.get_level_values(0) else pd.DataFrame()
    else:
        prices = raw
        volumes = pd.DataFrame()
    loader.save(raw, cache_path)

# Filter stocks with sufficient history
min_obs = MIN_HISTORY_MONTHS * 21
valid = prices.columns[prices.notna().sum() >= min_obs]
prices = prices[valid]
if not volumes.empty:
    volumes = volumes[volumes.columns.intersection(valid)]

print(f"Loaded: {prices.shape[1]} stocks \u00d7 {prices.shape[0]} trading days")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.head()

## 2. PCA — Covariance Decomposition

In [ ]:
preprocessor = Preprocessor()
log_returns = preprocessor.compute_log_returns(prices)
return_matrix = preprocessor.build_return_matrix(log_returns)

decomposer = CovarianceDecomposer(return_matrix)
decomposer.compute_covariance()
decomposer.eigendecompose()
decomposer.fit_pca(n_components=min(N_PCA_COMPONENTS, return_matrix.shape[1] - 1))

k_star = decomposer.select_components(variance_threshold=VARIANCE_THRESHOLD)
print(f"Selected {k_star} principal components explaining \u2265{VARIANCE_THRESHOLD*100:.0f}% variance")

fig, ax = plt.subplots(figsize=(10, 5))
decomposer.plot_scree(ax=ax)
plt.tight_layout()
plt.show()

## 3. K-Means Clustering

In [ ]:
clusterer = MarketClusterer(decomposer, n_clusters=N_CLUSTERS)
pc_scores = clusterer.get_stock_pc_scores(n_dims=3)
cluster_labels = clusterer.fit_kmeans()

# Silhouette analysis
sil_scores = clusterer.silhouette_analysis()
print("Silhouette scores by k:")
print(sil_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
clusterer.plot_clusters_2d(ax=axes[0])
sil_scores.plot(ax=axes[1], marker='o')
axes[1].set_title("Silhouette score vs number of clusters")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score")
plt.tight_layout(); plt.show()

## 4. Factor Engine

In [ ]:
engine = FactorEngine(prices=prices, volumes=volumes)
factor_data = engine.compute_all()

print("Factor data shape:", factor_data.shape)
print("\nFactor summary statistics (post winsorise + z-score):")
print(factor_data[['momentum_12_1', 'realized_vol', 'volume_trend']].describe().round(3))

## 5. Information Coefficient Analysis

In [ ]:
monthly_returns = preprocessor.compute_monthly_returns(prices)

# Compute forward 1-month returns
fwd_returns = monthly_returns.shift(-1)
fwd_long = fwd_returns.stack().reset_index()
fwd_long.columns = ['date', 'ticker', 'forward_return']

# Merge with factor data
factor_with_fwd = factor_data.merge(fwd_long, on=['date', 'ticker'], how='inner')

ic_analyzer = InformationCoefficient(
    factor_data=factor_with_fwd,
    forward_returns=factor_with_fwd[['date', 'ticker', 'forward_return']]
)

for factor in ['momentum_12_1', 'realized_vol', 'volume_trend']:
    summary = ic_analyzer.ic_summary(factor)
    print(f"\n{factor.upper()}")
    print(f"  Mean IC:    {summary['mean_ic']:.4f}")
    print(f"  IC Std:     {summary['ic_std']:.4f}")
    print(f"  IR:         {summary['ir']:.4f}")
    print(f"  t-stat:     {summary['ic_t_stat']:.2f}")
    print(f"  %% Positive: {summary['pct_positive']*100:.1f}%%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
for i, factor in enumerate(['momentum_12_1', 'realized_vol', 'volume_trend']):
    ic_analyzer.plot_ic_timeseries(factor, ax=axes[i])
plt.tight_layout(); plt.show()

## 6. EDA Dashboard

In [ ]:
from visualisation.eda import EDADashboard
eda = EDADashboard(factor_data=factor_data, prices=prices)
eda.factor_distributions()
plt.show()
eda.factor_correlation_heatmap()
plt.show()
eda.autocorrelation_decay()
plt.show()
eda.turnover_analysis()
plt.show()